In [10]:
import os
import smtplib
import pandas as pd
from email.message import EmailMessage
from PIL import Image, ImageDraw, ImageFont

# ==============================================================================
# CONFIGURATION
# ==============================================================================
SMTP_SERVER = 'smtp.gmail.com'
SMTP_PORT = 465
SENDER_EMAIL = 'isaacoluwaseyiajao@gmail.com'
SENDER_PASSWORD = 'autnmnmzputozewe'  # Replace with your 16-character Google App Password (NO spaces)

CONFERENCE_TITLE = "Data Analytics for Scientific Research & Innovation"
EVENT_DATE = "August 12, 2026"

# Ensure output directory exists
os.makedirs('certificates', exist_ok=True)


# ==============================================================================
# CERTIFICATE GENERATOR FUNCTION (WITH SIGNATURE IMAGES)
# ==============================================================================
def draw_certificate(participant_name, conference_title, date_str, output_pdf_path):
    width, height = 2400, 1700
    bg_color = (252, 252, 254)
    navy_blue = (0, 32, 96)
    accent_gold = (212, 175, 55)
    text_dark = (40, 40, 40)
    
    img = Image.new('RGB', (width, height), color=bg_color)
    draw = ImageDraw.Draw(img)

    # 1. Borders & Corner Elements
    draw.rectangle([50, 50, width - 50, height - 50], outline=navy_blue, width=12)
    draw.rectangle([70, 70, width - 70, height - 70], outline=accent_gold, width=4)

    draw.polygon([(50, 50), (220, 50), (50, 220)], fill=navy_blue)
    draw.polygon([(65, 65), (200, 65), (65, 200)], fill=accent_gold)
    
    draw.polygon([(width - 50, height - 50), (width - 220, height - 50), (width - 50, height - 220)], fill=navy_blue)
    draw.polygon([(width - 65, height - 65), (width - 200, height - 65), (width - 65, height - 200)], fill=accent_gold)

    # 2. Load Fonts
    try:
        font_header = ImageFont.truetype("georgiab.ttf", 48)
        font_cert = ImageFont.truetype("georgiab.ttf", 85)
        font_sub = ImageFont.truetype("arial.ttf", 36)
        font_name = ImageFont.truetype("georgiab.ttf", 75)
        font_body = ImageFont.truetype("arial.ttf", 38)
        font_conf = ImageFont.truetype("georgiab.ttf", 44)
        font_small = ImageFont.truetype("arial.ttf", 32)
    except IOError:
        font_header = font_cert = font_sub = font_name = font_body = font_conf = font_small = ImageFont.load_default()

    # 3. Header Text
    draw.text((width / 2, 170), "SCHOOL OF PURE AND APPLIED SCIENCES (SPAS)", fill=navy_blue, font=font_header, anchor="mm")
    draw.text((width / 2, 230), "FEDERAL POLYTECHNIC, ADO-EKITI", fill=text_dark, font=font_sub, anchor="mm")
    draw.line([(width / 2 - 300, 275), (width / 2 + 300, 275)], fill=accent_gold, width=3)

    # 4. Title & Recipient Name
    draw.text((width / 2, 370), "CERTIFICATE OF PARTICIPATION", fill=navy_blue, font=font_cert, anchor="mm")
    draw.text((width / 2, 480), "This is to certify that", fill=text_dark, font=font_sub, anchor="mm")
    draw.text((width / 2, 590), participant_name, fill=navy_blue, font=font_name, anchor="mm")
    draw.line([(width / 2 - 450, 640), (width / 2 + 450, 640)], fill=accent_gold, width=4)

    # 5. Event Description
    draw.text((width / 2, 720), "has actively participated in the workshop/conference on", fill=text_dark, font=font_body, anchor="mm")
    draw.text((width / 2, 810), f'"{conference_title}"', fill=navy_blue, font=font_conf, anchor="mm")
    draw.text((width / 2, 910), f"Held on {date_str}", fill=text_dark, font=font_body, anchor="mm")

    # 6. Gold Seal Badge
    badge_x, badge_y = width / 2, 1140
    badge_r = 85
    draw.ellipse([badge_x - badge_r, badge_y - badge_r, badge_x + badge_r, badge_y + badge_r], fill=accent_gold, outline=navy_blue, width=4)
    draw.ellipse([badge_x - badge_r + 10, badge_y - badge_r + 10, badge_x + badge_r - 10, badge_y + badge_r - 10], outline=(255, 255, 255), width=3)
    draw.text((badge_x, badge_y - 15), "SPAS", fill=navy_blue, font=font_sub, anchor="mm")
    draw.text((badge_x, badge_y + 25), "2026", fill=navy_blue, font=font_small, anchor="mm")

    # 7. Signatures Block
    # --- Left Signatory (Facilitator) ---
    draw.line([(350, 1420), (800, 1420)], fill=text_dark, width=2)
    draw.text((575, 1450), "Workshop Facilitator / Convener", fill=text_dark, font=font_small, anchor="mm")
    draw.text((575, 1490), "Dr. Isaac O. Ajao", fill=navy_blue, font=font_sub, anchor="mm")

    # Load & Paste Facilitator Signature Image
    try:
        sig_fac = Image.open("sig_facilitator.png").convert("RGBA")
        sig_fac = sig_fac.resize((280, 90))
        img.paste(sig_fac, (435, 1320), mask=sig_fac)
    except IOError:
        pass  # Proceeds without crashing if image is not present

    # --- Right Signatory (Committee Chairman) ---
    draw.line([(width - 800, 1420), (width - 350, 1420)], fill=text_dark, width=2)
    draw.text((width - 575, 1450), "Dean / Committee Chairman", fill=text_dark, font=font_small, anchor="mm")
    draw.text((width - 575, 1490), "School of Pure & Applied Sciences", fill=navy_blue, font=font_sub, anchor="mm")

    # Load & Paste Committee Chairman Signature Image
    try:
        sig_chair = Image.open("sig_chairman.png").convert("RGBA")
        sig_chair = sig_chair.resize((280, 90))
        img.paste(sig_chair, (width - 715, 1320), mask=sig_chair)
    except IOError:
        pass  # Proceeds without crashing if image is not present

    # 8. Save High-Res PDF
    img.save(output_pdf_path, "PDF", resolution=300.0)


# ==============================================================================
# BATCH EXECUTION & AUTOMATED MAILING LOOP
# ==============================================================================
# Read participants list from CSV
df = pd.read_csv('participants.csv')

print(f"Loaded {len(df)} participants from participants.csv\n")

# Connect to Gmail SMTP
with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
    server.login(SENDER_EMAIL, SENDER_PASSWORD)
    print("Authentication successful! Sending certificates with signatures...\n")

    for index, row in df.iterrows():
        name = row['Name']
        recipient_email = row['Email']

        # Clean name for filename
        clean_name = name.replace(' ', '_').replace('.', '')
        pdf_filename = f"certificates/Certificate_{clean_name}.pdf"

        # 1. Generate unique PDF certificate with signature overlays
        draw_certificate(
            participant_name=name,
            conference_title=CONFERENCE_TITLE,
            date_str=EVENT_DATE,
            output_pdf_path=pdf_filename
        )

        # 2. Construct Email Message
        msg = EmailMessage()
        msg['Subject'] = f'Certificate of Participation — {CONFERENCE_TITLE}'
        msg['From'] = SENDER_EMAIL
        msg['To'] = recipient_email
        msg.set_content(
            f"Dear {name},\n\n"
            f"Thank you for participating in the workshop/conference on '{CONFERENCE_TITLE}'.\n\n"
            f"Please find attached your official Certificate of Participation.\n\n"
            f"Best regards,\n"
            f"Dr. Isaac O. Ajao\n"
            f"Department of Statistics, Federal Polytechnic, Ado-Ekiti"
        )

        # 3. Attach PDF
        with open(pdf_filename, 'rb') as f:
            file_data = f.read()
            msg.add_attachment(file_data, maintype='application', subtype='pdf', filename=f"Certificate_{clean_name}.pdf")

        # 4. Send Email
        server.send_message(msg)
        print(f"✅ Generated & emailed signed certificate to: {name} ({recipient_email})")

print("\n🎉 All signed participant certificates processed and dispatched successfully!")

Loaded 3 participants from participants.csv

Authentication successful! Sending certificates with signatures...

✅ Generated & emailed signed certificate to: Dr Isaac Oluwaseyi Ajao (isaacoluwaseyiajao@gmail.com)
✅ Generated & emailed signed certificate to: Dr Seyi Ade (ajao_io@fedpolyado.edu.ng)
✅ Generated & emailed signed certificate to: Soft data (softdataconsult@gmail.com)

🎉 All signed participant certificates processed and dispatched successfully!
